# 02 — CNN TensorFlow: RGB, không augmentation
Huấn luyện từ đầu, cùng split và tiêu chí dev của KNN.

In [ ]:
from pathlib import Path
import os, sys
# Bỏ comment nếu Kaggle gắn nhiều bộ code hoặc dữ liệu.
# os.environ['CS231_PROJECT_ROOT'] = '/kaggle/input/TEN-DATASET-CODE'
# os.environ['CS231_DATA_ROOT'] = '/kaggle/input/TEN-DATASET-ANH/DATA'
# os.environ['CS231_CNN_PREVIOUS_RESULTS'] = '/kaggle/input/.../CS231_CNN/results_224'
os.environ.setdefault('MPLCONFIGDIR', '/tmp/cs231-matplotlib')
roots = [Path.cwd(), *Path.cwd().parents]
roots += [p for pattern in ('*', 'datasets/*/*') for p in Path('/kaggle/input').glob(pattern)]
if os.environ.get('CS231_PROJECT_ROOT'):
    roots = [Path(os.environ['CS231_PROJECT_ROOT'])]
projects = sorted({p.resolve() for p in roots if (p/'src/preprocessing.py').is_file()})
if len(projects) != 1:
    raise FileNotFoundError('Đặt CS231_PROJECT_ROOT trỏ tới thư mục chứa src.')
PROJECT_ROOT = projects[0]
sys.path.insert(0, str(PROJECT_ROOT))
from src.experiments.runtime import setup, restore_previous_output
DATA_ROOT, RESULTS_ROOT = setup('CNN', PROJECT_ROOT)

In [ ]:
restore_previous_output(RESULTS_ROOT, 'CNN', 'inputs/experiment_config.json')

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from IPython.display import display, Image
from src.CNN.data import CLASSES, CANDIDATES, load_config, configure_gpu, seed_everything
configure_gpu()
seed_everything(42)

## 1. Đọc dữ liệu và cấu hình
Không nạp toàn bộ ảnh vào RAM; tf.data đọc OpenCV theo batch, giữ đúng thứ tự khi đánh giá.

In [ ]:
protocol = load_config(RESULTS_ROOT)
CANDIDATE_ID = 'C01_RGB'
config = next(c for c in protocol['candidates'] if c['candidate_id'] == CANDIDATE_ID)
display(pd.DataFrame([config]))

## 2. Xây CNN bằng Sequential và model.add
Kiến trúc thật được khai báo trực tiếp bên dưới. Conv dùng Kaiming normal **fan_out**, BN dùng lớp TensorFlow tương thích BatchNorm2d của PyTorch: epsilon1e-5, trọng số thống kê mới0.1, running variance không chệch.
Hai Conv–BN–ReLU mỗi khối, MaxPool, SpatialDropout0.1; GAP → Dense128 → Dropout0.5 → Dense6 softmaxfloat32.
Augmentation OpenCV thực hiện trong dataset train trên ảnhuint8, trước khi chuyển xám và trước Rescaling(1/255). Inference không có augmentation.

In [ ]:
from src.CNN.model import TorchBatchNormalization, conv_initializer
# Conv: Kaiming normal fan_out; BatchNorm: thống kê như PyTorch.
# Augmentation OpenCV nằm trong dataset train trước grayscale/Rescaling.
model = models.Sequential(name='C01_RGB')
model.add(layers.Input(shape=(224, 224, 3), name='images'))
model.add(layers.Rescaling(1./255, dtype='float32', name='rescale'))
model.add(layers.Conv2D(32, (3, 3), padding='same', use_bias=False, kernel_initializer=conv_initializer(), name='conv1_1'))
model.add(TorchBatchNormalization(momentum=.1, epsilon=1e-5, name='bn1_1'))
model.add(layers.Activation('relu', name='relu1_1'))
model.add(layers.Conv2D(32, (3, 3), padding='same', use_bias=False, kernel_initializer=conv_initializer(), name='conv1_2'))
model.add(TorchBatchNormalization(momentum=.1, epsilon=1e-5, name='bn1_2'))
model.add(layers.Activation('relu', name='relu1_2'))
model.add(layers.MaxPooling2D((2, 2), name='pool1'))
model.add(layers.SpatialDropout2D(.1, name='drop1'))
model.add(layers.Conv2D(64, (3, 3), padding='same', use_bias=False, kernel_initializer=conv_initializer(), name='conv2_1'))
model.add(TorchBatchNormalization(momentum=.1, epsilon=1e-5, name='bn2_1'))
model.add(layers.Activation('relu', name='relu2_1'))
model.add(layers.Conv2D(64, (3, 3), padding='same', use_bias=False, kernel_initializer=conv_initializer(), name='conv2_2'))
model.add(TorchBatchNormalization(momentum=.1, epsilon=1e-5, name='bn2_2'))
model.add(layers.Activation('relu', name='relu2_2'))
model.add(layers.MaxPooling2D((2, 2), name='pool2'))
model.add(layers.SpatialDropout2D(.1, name='drop2'))
model.add(layers.Conv2D(128, (3, 3), padding='same', use_bias=False, kernel_initializer=conv_initializer(), name='conv3_1'))
model.add(TorchBatchNormalization(momentum=.1, epsilon=1e-5, name='bn3_1'))
model.add(layers.Activation('relu', name='relu3_1'))
model.add(layers.Conv2D(128, (3, 3), padding='same', use_bias=False, kernel_initializer=conv_initializer(), name='conv3_2'))
model.add(TorchBatchNormalization(momentum=.1, epsilon=1e-5, name='bn3_2'))
model.add(layers.Activation('relu', name='relu3_2'))
model.add(layers.MaxPooling2D((2, 2), name='pool3'))
model.add(layers.SpatialDropout2D(.1, name='drop3'))
model.add(layers.GlobalAveragePooling2D(name='embedding'))
model.add(layers.Dense(128, activation='relu', kernel_initializer='glorot_uniform', bias_initializer='zeros', name='dense128'))
model.add(layers.Dropout(.5, name='head_dropout'))
model.add(layers.Dense(6, activation='softmax', kernel_initializer='glorot_uniform', bias_initializer='zeros', dtype='float32', name='predictions'))

In [ ]:
# Display model summary
model.summary()

## 3. Huấn luyện và chọn checkpoint trên dev
AdamW + cosine LR theo epoch + global gradient clipping norm5, batch64, như protocol gốc. Bảng log ghi learning_rate thực tế từng epoch.
Sau mỗi epoch chỉ suy luận dev **một lần** để tính loss, Accuracy, Macro-F1 và MAE VND. Nhãn0..5 được đổi lại mệnh giá trước metric.
Lưu best.keras theo Macro-F1 dev, hòa thì MAE thấp hơn; vẫn hòa giữ epoch sớm. Lưu best.keras và history.csv. Bản học tập này chạy từ epoch1 khi gọi lại hàm huấn luyện.
Seed giúp chạy nhất quán hơn, nhưng không bảo đảm giống từng bit trên mọi máy. Không điều chỉnh tham số từ test.

In [ ]:
from src.CNN.pipeline import train_candidate
history, summary = train_candidate(model, CANDIDATE_ID, DATA_ROOT, RESULTS_ROOT)
display(history)
display(pd.DataFrame([{k:v for k,v in summary.items() if k != 'environment'}]))

## 4. Learning curve, cosine LR và ma trận nhầm lẫn
Bốn biểu đồ: loss, Accuracy, Macro-F1 dev và learning rate. Train có dropout/augmentation, dev dùng inference; đọc gap cùng độ dao động, không suy chỉ một nguyên nhân.
Hình, CSV, kiến trúc và checkpoint nằm trong candidates/<cấu hình>. Thời gian train và dev được ghi riêng.

In [ ]:
RUN = RESULTS_ROOT/'candidates'/CANDIDATE_ID
display(Image(filename=str(RUN/'learning_curves.png')))
display(pd.read_csv(RUN/'per_class_dev.csv'))
display(Image(filename=str(RUN/'confusion_matrix_dev.png')))

## 5. Giữ Output để notebook kế tiếp sử dụng